# 02 Algorithmic Foundations for Computational Biology

This notebook introduces core algorithmic ideas that support computational biology.

It connects selected Rosalind **Algorithmic Heights** problems to biological sequence analysis, including recurrence, binary search, sorting, merge sort, inversion counting, graph traversal, and connected components.

This notebook supports:

- **Module 01: Algorithmic Foundations — Arrays, Sorting, Searching, and Recurrence**
- **Module 02: Algorithmic Foundations — Graphs**

## Learning Goals

After completing this notebook, a learner should be able to:

- understand recurrence relations through Fibonacci-style examples;
- implement binary search;
- merge sorted arrays;
- implement merge sort;
- count inversions using divide-and-conquer;
- represent graphs using adjacency lists;
- perform breadth-first search;
- compute connected components;
- understand why these algorithmic tools matter for computational biology.

## Connection to Original Rosalind Solutions

This notebook is connected to my original Rosalind **Algorithmic Heights** solutions preserved under:

- `original_rosalind_tracks/algorithmic_heights/`

The examples here are rewritten as teaching-oriented reference implementations. They are intended to explain the algorithmic ideas behind selected problems such as `FIBO`, `BINS`, `MER`, `MS`, `INV`, `DEG`, `BFS`, and `CC`, rather than directly copying the original solution files.

## 1. Recurrence relations

This connects to Rosalind problem `FIBO`.

A recurrence relation defines a value using previous values. In computational biology, recurrence appears in dynamic programming, population modelling, RNA structure counting, and sequence alignment.

In [ ]:
def fibonacci(n: int) -> int:
    """Return the nth Fibonacci number using an iterative recurrence."""
    if n <= 0:
        raise ValueError("n must be positive")

    if n == 1 or n == 2:
        return 1

    previous = 1
    current = 1

    for _ in range(3, n + 1):
        previous, current = current, previous + current

    return current


for n in range(1, 11):
    print(n, fibonacci(n))

## 2. Binary search

This connects to Rosalind problem `BINS`.

Binary search works on a sorted list. It repeatedly cuts the search interval in half.

In [ ]:
def binary_search(values: list[int], target: int) -> int:
    """Return the zero-based index of target in values, or -1 if not found."""
    left = 0
    right = len(values) - 1

    while left <= right:
        middle = (left + right) // 2

        if values[middle] == target:
            return middle

        if values[middle] < target:
            left = middle + 1
        else:
            right = middle - 1

    return -1


numbers = [1, 3, 5, 7, 9, 11, 13]

print(binary_search(numbers, 7))
print(binary_search(numbers, 4))

### Rosalind-style one-based output

Some Rosalind problems expect one-based positions. Python uses zero-based indexing. It is useful to keep the core function Pythonic and convert positions only when needed.

In [ ]:
def to_one_based(index: int) -> int:
    """Convert a zero-based index to a one-based index, preserving -1 for not found."""
    if index == -1:
        return -1

    return index + 1


print(to_one_based(binary_search(numbers, 7)))
print(to_one_based(binary_search(numbers, 4)))

## 3. Merging sorted arrays

This connects to Rosalind problem `MER`.

Merging is the core operation inside merge sort. It is also useful whenever two ordered biological or numerical data streams need to be combined.

In [ ]:
def merge_sorted(left: list[int], right: list[int]) -> list[int]:
    """Merge two sorted lists into one sorted list."""
    merged = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])

    return merged


a = [1, 4, 7, 10]
b = [2, 3, 8, 11]

print(merge_sorted(a, b))

## 4. Merge sort

This connects to Rosalind problem `MS`.

Merge sort uses divide-and-conquer:

1. split the list into halves;
2. sort each half recursively;
3. merge the sorted halves.

In [ ]:
def merge_sort(values: list[int]) -> list[int]:
    """Return a sorted copy of values using merge sort."""
    if len(values) <= 1:
        return values

    middle = len(values) // 2
    left = merge_sort(values[:middle])
    right = merge_sort(values[middle:])

    return merge_sorted(left, right)


unsorted_values = [8, 3, 5, 1, 9, 2, 7]
print(merge_sort(unsorted_values))

## 5. Counting inversions

This connects to Rosalind problem `INV`.

An inversion is a pair of positions `(i, j)` where:

- `i < j`
- `values[i] > values[j]`

Inversions measure how far a list is from being sorted. Similar ordering ideas appear in genome rearrangement and comparative sequence ordering problems.

In [ ]:
def count_inversions(values: list[int]) -> tuple[list[int], int]:
    """Return a sorted copy of values and the number of inversions."""
    if len(values) <= 1:
        return values, 0

    middle = len(values) // 2
    left, left_inversions = count_inversions(values[:middle])
    right, right_inversions = count_inversions(values[middle:])

    merged = []
    split_inversions = 0

    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            split_inversions += len(left) - i
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])

    total_inversions = left_inversions + right_inversions + split_inversions
    return merged, total_inversions


values = [2, 3, 8, 6, 1]
sorted_values, inversion_count = count_inversions(values)

print("Sorted:", sorted_values)
print("Inversions:", inversion_count)

## 6. Graph representation

This connects to Rosalind graph problems such as `DEG`, `DDEG`, `BFS`, and `CC`.

A graph can be represented as an adjacency list: each node maps to a list of neighboring nodes.

In [ ]:
graph = {
    1: [2, 3],
    2: [1, 4],
    3: [1],
    4: [2],
    5: [6],
    6: [5],
}

graph

## 7. Degree array

This connects to Rosalind problem `DEG`.

The degree of a node is the number of edges connected to it.

In [ ]:
def degree_array(graph: dict[int, list[int]]) -> dict[int, int]:
    """Return the degree of each node in an undirected graph."""
    return {node: len(neighbors) for node, neighbors in graph.items()}


degree_array(graph)

## 8. Breadth-first search

This connects to Rosalind problem `BFS`.

Breadth-first search explores a graph level by level. It can be used to compute shortest unweighted path distances from a starting node.

In [ ]:
from collections import deque


def bfs_distances(graph: dict[int, list[int]], start: int) -> dict[int, int]:
    """Return shortest unweighted distances from start to all reachable nodes."""
    distances = {node: -1 for node in graph}
    distances[start] = 0

    queue = deque([start])

    while queue:
        current = queue.popleft()

        for neighbor in graph[current]:
            if distances[neighbor] == -1:
                distances[neighbor] = distances[current] + 1
                queue.append(neighbor)

    return distances


bfs_distances(graph, start=1)

## 9. Connected components

This connects to Rosalind problem `CC`.

A connected component is a group of nodes where every node can be reached from every other node in the same group.

In [ ]:
def connected_components(graph: dict[int, list[int]]) -> list[list[int]]:
    """Return connected components of an undirected graph."""
    visited = set()
    components = []

    for node in graph:
        if node in visited:
            continue

        component = []
        queue = deque([node])
        visited.add(node)

        while queue:
            current = queue.popleft()
            component.append(current)

            for neighbor in graph[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        components.append(component)

    return components


connected_components(graph)

## 10. Why algorithms matter in computational biology

The same algorithmic ideas reappear across computational biology.

| Algorithmic Idea | Bioinformatics Connection |
|---|---|
| Recurrence | dynamic programming, population growth, RNA structure counting |
| Binary search | efficient lookup in sorted biological data |
| Sorting | k-mer organization, suffix arrays, preprocessing |
| Merge sort | divide-and-conquer sequence/data processing |
| Inversion counting | ordering differences, rearrangement-style reasoning |
| Graph representation | overlap graphs, de Bruijn graphs, phylogenetic trees |
| BFS and connectivity | graph traversal, assembly graphs, biological networks |

## 11. Mini exercise set

Try modifying the functions above to solve these small exercises.

1. Modify `fibonacci()` to return the full sequence up to `n`.
2. Modify `binary_search()` to return all positions if duplicates exist.
3. Write a function that checks whether a list is sorted.
4. Write a function that computes the number of connected components only, instead of returning each component.
5. Create a graph from an edge list such as `[(1, 2), (1, 3), (4, 5)]`.
6. Use `bfs_distances()` to find unreachable nodes from a given starting node.

## Summary

This notebook introduced algorithmic foundations that support later computational biology modules.

The key message is that bioinformatics is not only about biological facts. It also depends heavily on efficient algorithms for processing strings, arrays, graphs, and structured data.

These foundations prepare the learner for:

- dynamic programming and sequence alignment;
- string matching and genome indexing;
- graph-based genome assembly;
- phylogenetics and tree-based reasoning;
- ML-ready biological feature construction.